# 小红书评论文本数据分析：第三部分\n\n本 Notebook 对应课程设计第三部分：数据预处理与变量构建。\n\n你今晚要完成的核心结果：\n- 清洗前后样本量对比\n- 情感得分变量\n- 文本长度、表情、配图、时间时段等衍生变量\n- 点赞量对数变换变量\n

In [ ]:
from __future__ import annotations\n\nfrom pathlib import Path\nimport math\nimport re\n\nimport numpy as np\nimport pandas as pd\n\nDATA_DIR = Path('/Users/xinxinhuashe/Documents/1/python课/data')\nOUTPUT_DIR = Path('/Users/xinxinhuashe/Documents/1/python课/output')\nOUTPUT_DIR.mkdir(parents=True, exist_ok=True)\n\nDATA_DIR\n

## 1. 读入数据\n\n先把老师给的数据文件放进 `data/` 文件夹，再修改下面文件名。\n

In [ ]:
file_name = '请把数据文件名改成这里.csv'\nfile_path = DATA_DIR / file_name\n\nif file_path.suffix.lower() == '.csv':\n    df = pd.read_csv(file_path)\nelif file_path.suffix.lower() in {'.xlsx', '.xls'}:\n    df = pd.read_excel(file_path)\nelse:\n    raise ValueError('请把 file_name 改成 csv 或 xlsx 文件名')\n\ndf.head()\n

## 2. 查看字段并手动确认列名\n\n这一格先运行，看看你的实际字段长什么样。\n

In [ ]:
print('数据维度:', df.shape)\nprint('字段列表:')\nfor col in df.columns:\n    print('-', col)\n

In [ ]:
# 把下面 4 个列名改成你自己数据里的实际列名\ntext_col = '评论内容'\nlike_col = '点赞数'\ntime_col = '评论时间'\nimage_col = '图片数量'\n\nuse_cols = [c for c in [text_col, like_col, time_col, image_col] if c in df.columns]\ndf = df.copy()\ndf[use_cols].head()\n

## 3. 原始数据清洗\n\n这里按照课程设计需要，依次处理：空值、重复值、灌水数据。\n

In [ ]:
clean_df = df.copy()\ninitial_n = len(clean_df)\n\nclean_df[text_col] = clean_df[text_col].astype(str).str.strip()\nempty_mask = clean_df[text_col].isin(['', 'nan', 'None'])\nempty_n = int(empty_mask.sum())\nclean_df = clean_df.loc[~empty_mask].copy()\n\ndup_mask = clean_df.duplicated(subset=[text_col], keep='first')\ndup_n = int(dup_mask.sum())\nclean_df = clean_df.loc[~dup_mask].copy()\n\nwater_mask = (\n    clean_df[text_col].str.len().fillna(0).lt(2)\n    | clean_df[text_col].str.fullmatch(r'[\\W_]+', na=False)\n    | clean_df[text_col].str.fullmatch(r'(哈)+|(啊)+|(哦)+', na=False)\n)\nwater_n = int(water_mask.sum())\nclean_df = clean_df.loc[~water_mask].copy()\n\nsummary_clean = pd.DataFrame({\n    '处理步骤': ['原始样本', '删除空值后', '删除重复值后', '删除灌水数据后'],\n    '删除数量': [0, empty_n, dup_n, water_n],\n    '保留数量': [\n        initial_n,\n        initial_n - empty_n,\n        initial_n - empty_n - dup_n,\n        len(clean_df),\n    ],\n})\n\nsummary_clean\n

## 4. 情感量化处理\n\n这里先给你一个能直接交作业的基础版本。后续如果你想升级成更正式的模型，我们再替换。\n

In [ ]:
positive_words = {'喜欢', '推荐', '不错', '满意', '好看', '喜欢死了', '好用', '值得', '真实', '开心'}\nnegative_words = {'不好', '失望', '踩雷', '一般', '避雷', '难用', '无语', '后悔', '差', '生气'}\n\ndef sentiment_score(text: str) -> float:\n    text = str(text)\n    pos = sum(word in text for word in positive_words)\n    neg = sum(word in text for word in negative_words)\n    return pos - neg\n\nclean_df['sentiment_score'] = clean_df[text_col].apply(sentiment_score)\nclean_df[['sentiment_score', text_col]].head()\n

## 5. 构建衍生变量\n

In [ ]:
emoji_pattern = re.compile('[😀-🙏🌀-🗿🚀-🛿🇠-🇿✨❤🥹🤣😭😂😅]')\n\nclean_df['text_length'] = clean_df[text_col].astype(str).str.len()\nclean_df['has_emoji'] = clean_df[text_col].astype(str).apply(lambda x: int(bool(emoji_pattern.search(x))))\n\nif image_col in clean_df.columns:\n    clean_df['has_image'] = clean_df[image_col].fillna(0).apply(lambda x: int(float(x) > 0))\nelse:\n    clean_df['has_image'] = np.nan\n\nif time_col in clean_df.columns:\n    clean_df[time_col] = pd.to_datetime(clean_df[time_col], errors='coerce')\n    clean_df['hour'] = clean_df[time_col].dt.hour\n\n    def get_period(hour):\n        if pd.isna(hour):\n            return '未知'\n        if 0 <= hour < 6:\n            return '凌晨'\n        if 6 <= hour < 12:\n            return '上午'\n        if 12 <= hour < 18:\n            return '下午'\n        return '晚上'\n\n    clean_df['time_period'] = clean_df['hour'].apply(get_period)\nelse:\n    clean_df['time_period'] = '未知'\n\nclean_df[[text_col, 'text_length', 'has_emoji', 'has_image', 'time_period']].head()\n

## 6. 点赞量对数变换\n

In [ ]:
clean_df[like_col] = pd.to_numeric(clean_df[like_col], errors='coerce').fillna(0)\nclean_df['log_likes'] = np.log1p(clean_df[like_col])\nclean_df[[like_col, 'log_likes']].describe()\n

## 7. 输出第三部分可以直接写进报告的统计结果\n

In [ ]:
report_stats = {\n    '原始样本量': initial_n,\n    '清洗后样本量': len(clean_df),\n    '删除空值数量': empty_n,\n    '删除重复值数量': dup_n,\n    '删除灌水数量': water_n,\n    '情感得分均值': round(clean_df['sentiment_score'].mean(), 4),\n    '情感得分最小值': clean_df['sentiment_score'].min(),\n    '情感得分最大值': clean_df['sentiment_score'].max(),\n    '文本长度均值': round(clean_df['text_length'].mean(), 2),\n    '文本长度最小值': clean_df['text_length'].min(),\n    '文本长度最大值': clean_df['text_length'].max(),\n}\n\nreport_stats\n

In [ ]:
summary_clean.to_csv(OUTPUT_DIR / '第三部分_清洗结果汇总.csv', index=False)\nclean_df.to_csv(OUTPUT_DIR / '第三部分_清洗后数据.csv', index=False)\npd.DataFrame([report_stats]).to_csv(OUTPUT_DIR / '第三部分_描述统计结果.csv', index=False)\nprint('已导出第三部分结果文件。')\n

## 8. 你最后要复制到 PPT/论文里的话\n\n第三部分建议保留三类结果：\n- 清洗流程和样本保留情况\n- 变量定义表\n- 关键描述性统计结果\n\n如果你把数据放进 `data/` 后告诉我字段名，我可以继续帮你把这个 Notebook 调成完全匹配你数据的一版。\n